In [2]:
import mdtraj as md
import numpy as np

In [75]:
amino_dict = {
    'ALA': 0,
    'ARG': 1,
    'ASN': 2,
    'ASP': 3,
    'CYS': 4,
    'GLN': 5,
    'GLU': 6,
    'GLY': 7,
    'HIS': 8,
    'ILE': 9,
    'LEU': 10,
    'LYS': 11,
    'MET': 12,
    'PHE': 13,
    'PRO': 14,
    'SER': 15,
    'THR': 16,
    'TRP': 17,
    'TYR': 18,
    'VAL': 19
}

In [157]:
def compute_angle(a: np.ndarray, b: np.ndarray) -> float:
    dot = np.dot(a, b)
    dot = np.clip(dot, -1.0, 1.0)
    return np.arccos(dot)

def compute_length(a: np.ndarray, b: np.ndarray) -> float:
    return np.linalg.norm(a - b)

def compute_bond_angle(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> float:
    r1 = a - b
    r2 = c - b
    r1 /= np.linalg.norm(r1)
    r2 /= np.linalg.norm(r2)
    return compute_angle(r1, r2)

def compute_dihedral_angle(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> float:
    b1 = a - b
    b2 = c - b
    b3 = c - d

    c1 = np.cross(b2, b3)
    c2 = np.cross(b1, b2)

    p1 = np.dot(b1, c1) * np.linalg.norm(b2)
    p2 = np.dot(c1, c2)

    return np.arctan2(p1, p2)

def compute_plane_vector(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> np.ndarray:
    ba = b - a
    bc = b - c
    normal = np.cross(ba, bc)
    return normal / np.linalg.norm(normal)

def compute_bisect_vector(a: np.ndarray, b: np.ndarray, c: np.ndarray) -> np.ndarray:
    ba = b - a
    bc = b - c
    ba /= np.linalg.norm(ba)
    bc /= np.linalg.norm(bc)
    bisect = (ba + bc) / 2.0
    return bisect / np.linalg.norm(bisect)

def compute_plane_angle(n1, ca1, o1, n2, ca2, o2) -> float:
    normal1 = compute_plane_vector(n1, ca1, o1)
    normal2 = compute_plane_vector(n2, ca2, o2)
    return compute_angle(normal1, normal2)

def compute_bisect_angle(n1, ca1, o1, n2, ca2, o2) -> float:
    bisect1 = compute_bisect_vector(n1, ca1, o1)
    bisect2 = compute_bisect_vector(n2, ca2, o2)
    return compute_angle(bisect1, bisect2)

def compute_delta_dihedral_angle(a, b, c, d_old, d_new) -> float:
    angle_old = compute_dihedral_angle(a, b, c, d_old)
    angle_new = compute_dihedral_angle(a, b, c, d_new)
    return angle_new - angle_old

def compute_center(atoms: list[np.ndarray]) -> np.ndarray:
    return sum(atoms) / len(atoms)

def normalize_angle(angle: float) -> float:
    angle = angle % 360.0
    return angle if angle >= 0 else angle + 360.0

def angle2index(angle: float, bin_size: float) -> int:
    #angle = np.degrees(angle) + 180.0
    #return int(((angle + 15.0) % 360.0) / bin_size)
    return int(angle / bin_size)

In [77]:
traj = md.load('/n/home01/kibumpark/pkg/dbfold2/examples/chignolin/1uao_processed.pdb')
traj = traj.atom_slice(traj.top.select('backbone'))

In [78]:
phis = (md.compute_phi(traj)[1][0][:-1] + np.pi)* 180/np.pi
psis = (md.compute_psi(traj)[1][0][1:] + np.pi)* 180/np.pi
print(f'phi: {phis}')
print(f'psi: {psis}')

phi: [ 95.16831  108.29828  104.038185 119.724335  72.95174  270.81134
  19.303999 110.83908 ]
psi: [277.90628 295.19165 167.16786 140.63492 157.53531 214.06691 345.35364
 287.1073 ]


In [79]:
plane_vectors = []
bisect_vectors = []
for i in range(10):
    n = traj.xyz[0][i*4]
    ca = traj.xyz[0][i*4 + 1]
    o = traj.xyz[0][i*4 + 3]
    plane_vectors.append(compute_plane_vector(n, ca, o))
    bisect_vectors.append(compute_bisect_vector(n, ca, o))

In [80]:
for i in range(10):
    print(traj.top.atom(i*4).name, traj.top.atom(i*4 + 1).name, traj.top.atom(i*4 + 3).name)

N CA O
N CA O
N CA O
N CA O
N CA O
N CA O
N CA O
N CA O
N CA O
N CA O


In [81]:
plane_angles = []
bisect_angles = []
for i in range(1,9):
    plane_angles.append(compute_angle(plane_vectors[i-1], plane_vectors[i+1]))
    bisect_angles.append(compute_angle(bisect_vectors[i-1], bisect_vectors[i+1]))
plane_angles = np.array(plane_angles) * 180 / np.pi
bisect_angles = np.array(bisect_angles) * 180 / np.pi
print("Plane angles:", plane_angles)
print("Bisect angles:", bisect_angles)

Plane angles: [101.0913549   89.41513163  30.25780757 116.20004355 100.64396178
 103.43609894  97.5256055  116.90883932]
Bisect angles: [ 90.18838091  89.52935967   9.09392427 165.26512994  93.74556609
  90.26837864  66.03688979 157.14893801]


In [82]:
def load_data(filename):
    data_map = {}
    with open(filename, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) >= 8:
                key = tuple(map(int, parts[:7]))  # First 7 columns
                value = int(parts[7])             # 8th column
                data_map[key] = value
    return data_map

def lookup_value(data_map, col1, col2, col3, col4, col5, col6, col7):
    key = (col1, col2, col3, col4, col5, col6, col7)
    return data_map.get(key, None)

In [83]:
filename = '/n/home01/kibumpark/pkg/dbfold2/MCPU2/config_files/triple.energy'
data = load_data(filename)

In [84]:
res_types = []
for res_name in traj.top.to_dataframe()[0][traj.top.to_dataframe()[0]['name'] == 'CA']['resName']:
    res_types.append(amino_dict[res_name])

In [ ]:
[7, 18, 3, 14, 6, 16, 7, 16, 17, 7]
e_tot = 0
for i in range(8):
    plane_index = int(plane_angles[i] // 30)
    bisect_index = int(bisect_angles[i] // 30)
    phi_index = int(phis[i] // 60)
    psi_index = int(psis[i] // 60)
    print(f"Residue {i+1}: {res_types[i], res_types[i+1], res_types[i+2]} Plane Index: {plane_index}, Bisect Index: {bisect_index}, Phi Index: {phi_index}, Psi Index: {psi_index}")
    result = lookup_value(data, res_types[i], res_types[i+1], res_types[i+2], plane_index, bisect_index, phi_index, psi_index)
    if result is None:
        result = 1000 
    e_tot += result
    print("Result:", result)
print("Total Energy:", e_tot/1000)

Residue 1: (7, 18, 3) Plane Index: 3, Bisect Index: 3, Phi Index: 1, Psi Index: 4
Result: 441
Residue 2: (18, 3, 14) Plane Index: 2, Bisect Index: 2, Phi Index: 1, Psi Index: 4
Result: -856
Residue 3: (3, 14, 6) Plane Index: 1, Bisect Index: 0, Phi Index: 1, Psi Index: 2
Result: -844
Residue 4: (14, 6, 16) Plane Index: 3, Bisect Index: 5, Phi Index: 1, Psi Index: 2
Result: -852
Residue 5: (6, 16, 7) Plane Index: 3, Bisect Index: 3, Phi Index: 1, Psi Index: 2
Result: -741
Residue 6: (16, 7, 16) Plane Index: 3, Bisect Index: 3, Phi Index: 4, Psi Index: 3
Result: -6
Residue 7: (7, 16, 17) Plane Index: 3, Bisect Index: 2, Phi Index: 0, Psi Index: 5
Result: -568
Residue 8: (16, 17, 7) Plane Index: 3, Bisect Index: 5, Phi Index: 1, Psi Index: 4
Result: 1000
Total Energy: -2.426


In [90]:
traj = md.load('/n/home01/kibumpark/pkg/dbfold2/examples/chignolin/1uao_processed.pdb')

In [136]:
chi_angles = np.zeros((traj.n_residues, 4))
chi_indices = np.zeros((traj.n_residues, 4), dtype=int)

In [149]:
import math
def normalize_angle(ang):
    # Bring angle into (-180, 180)
    while ang < -180.0:
        ang += 360.0
    while ang >= 180.0:
        ang -= 360.0
    # Shift angle into (0, 360) with slight bias
    if ang < 0:
        ang += 180.00001
    else:
        ang += 179.99999
    # Add 15 and wrap into [0, 360)
    ang += 15.0
    ang = math.fmod(ang, 360.0)
    if ang < 0:
        ang += 360.0  # ensure positive result from fmod
    return ang

In [172]:
atoms, chi1 = md.compute_chi1(traj)
for i, atom in enumerate(atoms):
    residue = list(traj.top.atoms)[atom[0]].residue
    res_name = residue.name
    res_index = residue.index
    #chi_angles[res_index, 0] = chi1[0][i] * 180 / np.pi
    print(f"Residue {res_index} Chi1: {[list(traj.top.atoms)[a].name for a in atom]}")
    chi_angles[res_index, 0] = normalize_angle(chi1[0][i] * 180 / np.pi)
    chi_indices[res_index, 0] = angle2index(chi_angles[res_index, 0], 15)
atoms, chi2 = md.compute_chi2(traj)
for i, atom in enumerate(atoms):
    residue = list(traj.top.atoms)[atom[0]].residue
    res_name = residue.name
    res_index = residue.index
    #chi_angles[res_index, 1] = chi2[0][i] * 180 / np.pi
    print(f"Residue {res_index} Chi2: {[list(traj.top.atoms)[a].name for a in atom]}")
    chi_angles[res_index, 1] = normalize_angle(chi2[0][i] * 180 / np.pi)
    chi_indices[res_index, 1] = angle2index(chi_angles[res_index, 1], 15)
atoms, chi3 = md.compute_chi3(traj)
for i, atom in enumerate(atoms):
    residue = list(traj.top.atoms)[atom[0]].residue
    res_name = residue.name
    res_index = residue.index
    #chi_angles[res_index, 2] = chi3[0][i] * 180 / np.pi
    print(f"Residue {res_index} Chi3: {[list(traj.top.atoms)[a].name for a in atom]}")
    chi_angles[res_index, 2] = normalize_angle(chi3[0][i] * 180 / np.pi)
    chi_indices[res_index, 2] = angle2index(chi_angles[res_index, 2], 15)
atoms, chi4 = md.compute_chi4(traj)
for i, atom in enumerate(atoms):
    residue = list(traj.top.atoms)[atom[0]].residue
    res_name = residue.name
    res_index = residue.index
    #chi_angles[res_index, 3] = chi4[0][i] * 180 / np.pi
    print(f"Residue {res_index} Chi4: {[list(traj.top.atoms)[a].name for a in atom]}")
    chi_angles[res_index, 3] = normalize_angle(chi4[0][i] * 180 / np.pi)
    chi_indices[res_index, 3] = angle2index(chi_angles[res_index, 3], 15)
print("Chi angles:")
for i in range(10):
    print(f"Residue {i+1}: {chi_angles[i, 0]:.2f}, {chi_angles[i, 1]:.2f}, {chi_angles[i, 2]:.2f}, {chi_angles[i, 3]:.2f}")
print("Chi indices:")
for i in range(1, 9):
    print(f"Residue {i+1}: {chi_indices[i, 0]}, {chi_indices[i, 1]}, {chi_indices[i, 2]}, {chi_indices[i, 3]}")

Residue 1 Chi1: ['N', 'CA', 'CB', 'CG']
Residue 2 Chi1: ['N', 'CA', 'CB', 'CG']
Residue 3 Chi1: ['N', 'CA', 'CB', 'CG']
Residue 4 Chi1: ['N', 'CA', 'CB', 'CG']
Residue 5 Chi1: ['N', 'CA', 'CB', 'OG1']
Residue 7 Chi1: ['N', 'CA', 'CB', 'OG1']
Residue 8 Chi1: ['N', 'CA', 'CB', 'CG']
Residue 1 Chi2: ['CA', 'CB', 'CG', 'CD1']
Residue 2 Chi2: ['CA', 'CB', 'CG', 'OD1']
Residue 3 Chi2: ['CA', 'CB', 'CG', 'CD']
Residue 4 Chi2: ['CA', 'CB', 'CG', 'CD']
Residue 8 Chi2: ['CA', 'CB', 'CG', 'CD1']
Residue 4 Chi3: ['CB', 'CG', 'CD', 'OE1']
Chi angles:
Residue 1: 0.00, 0.00, 0.00, 0.00
Residue 2: 180.15, 86.56, 0.00, 0.00
Residue 3: 184.90, 57.49, 0.00, 0.00
Residue 4: 22.20, 338.22, 0.00, 0.00
Residue 5: 203.94, 252.98, 27.54, 0.00
Residue 6: 74.77, 0.00, 0.00, 0.00
Residue 7: 0.00, 0.00, 0.00, 0.00
Residue 8: 15.53, 0.00, 0.00, 0.00
Residue 9: 323.64, 94.57, 0.00, 0.00
Residue 10: 0.00, 0.00, 0.00, 0.00
Chi indices:
Residue 2: 12, 5, 0, 0
Residue 3: 12, 3, 0, 0
Residue 4: 1, 22, 0, 0
Residue 5: 13,

In [169]:
atoms

array([[ 4,  5,  8,  9],
       [16, 17, 20, 21],
       [24, 25, 28, 29],
       [31, 32, 35, 36],
       [40, 41, 44, 46],
       [51, 52, 55, 57],
       [58, 59, 62, 63]])

In [164]:
filename = '/n/home01/kibumpark/pkg/dbfold2/MCPU2/config_files/sct.energy'
sc_data = load_data(filename)

In [173]:
e_tot = 0
for i in range(8):
    print(f"Residue {i+1}: {res_types[i], res_types[i+1], res_types[i+2]} Chi Indices: {chi_indices[i+1, 0]}, {chi_indices[i+1, 1]}, {chi_indices[i+1, 2]}, {chi_indices[i+1, 3]}")
    result = lookup_value(sc_data, res_types[i], res_types[i+1], res_types[i+2], chi_indices[i+1][0], chi_indices[i+1][1], chi_indices[i+1][2], chi_indices[i+1][3])
    if result is None:
        result = 1000 
    e_tot += result
    print("Result:", result)
print("Total Energy:", e_tot/1000)

Residue 1: (7, 18, 3) Chi Indices: 12, 5, 0, 0
Result: 1000
Residue 2: (18, 3, 14) Chi Indices: 12, 3, 0, 0
Result: 1000
Residue 3: (3, 14, 6) Chi Indices: 1, 22, 0, 0
Result: 1000
Residue 4: (14, 6, 16) Chi Indices: 13, 16, 1, 0
Result: 1000
Residue 5: (6, 16, 7) Chi Indices: 4, 0, 0, 0
Result: -926
Residue 6: (16, 7, 16) Chi Indices: 0, 0, 0, 0
Result: 1000
Residue 7: (7, 16, 17) Chi Indices: 1, 0, 0, 0
Result: 1000
Residue 8: (16, 17, 7) Chi Indices: 21, 6, 0, 0
Result: 1000
Total Energy: 6.074


In [174]:
lookup_value(sc_data, 0, 1, 0, 0, 0, 0, 0)  # Example lookup

-164